# Training Machine Learning Models for Maize Leaf Image Classification

Retrieving images from https://github.com/vlattko/nitro/tree/main

## EDA on dataset

In [2]:
from pathlib import Path
import re
from collections import Counter
import pandas as pd

root = Path("../data/images_proximal")

# Count images per class from filename pattern: N{X} (YY).JPG
pattern = re.compile(r"^N(?P<x>\d+)\s*\(\d+\)\.jpe?g$", re.IGNORECASE)

class_counts = Counter()
unmatched_files = []
image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

for p in root.rglob("*"):
    if p.is_file() and p.suffix.lower() in image_exts:
        m = pattern.match(p.name)
        if m:
            class_counts[f"N{m.group('x')}"] += 1
        else:
            unmatched_files.append(p)

df_name_counts = (
    pd.DataFrame(sorted(class_counts.items(), key=lambda t: int(t[0][1:])), columns=["class", "count"])
    if class_counts
    else pd.DataFrame(columns=["class", "count"])
)

display(df_name_counts)
print(f"Total matched images: {int(df_name_counts['count'].sum()) if not df_name_counts.empty else 0}")
print(f"Files not matching pattern: {len(unmatched_files)}")

,class,count
0,N0,400
1,N75,400


Total matched images: 800
Files not matching pattern: 400


400 images per class -> N0, N75 and NFull

## Divide Test Images into Train/Val

In [3]:
from sklearn.model_selection import train_test_split
import shutil

# Create train and val directory structure
train_root = Path("../data/train")
val_root = Path("../data/val")

# Define all classes (including NFull from unmatched_files)
classes = ['N0', 'N75', 'NFull']

# Create directories
for split_root in [train_root, val_root]:
    for cls in classes:
        (split_root / cls).mkdir(parents=True, exist_ok=True)

# Process matched files (N0 and N75)
for cls in ['N0', 'N75']:
    cls_files = [p for p in root.rglob("*") if p.is_file() and pattern.match(p.name) and pattern.match(p.name).group('x') == cls[1:]]
    
    # Split 80:20
    train_files, val_files = train_test_split(cls_files, test_size=0.2, random_state=42)
    
    # Copy to train
    for f in train_files:
        shutil.copy2(f, train_root / cls / f.name)
    
    # Copy to val
    for f in val_files:
        shutil.copy2(f, val_root / cls / f.name)
    
    print(f"{cls}: {len(train_files)} train, {len(val_files)} val")

# Process NFull files (unmatched_files)
train_files, val_files = train_test_split(unmatched_files, test_size=0.2, random_state=42)

for f in train_files:
    shutil.copy2(f, train_root / 'NFull' / f.name)

for f in val_files:
    shutil.copy2(f, val_root / 'NFull' / f.name)

print(f"NFull: {len(train_files)} train, {len(val_files)} val")

N0: 320 train, 80 val
N75: 320 train, 80 val
NFull: 320 train, 80 val
